## Example 1: Basic R-tree index

In [ ]:
# Example 1: Basic R-tree index
from rtree import index

print("Testing rtree basic index:")

# Create R-tree index
idx = index.Index()
print(f"✓ Index created: {type(idx).__name__}")

# Insert spatial objects (id, (min_x, min_y, max_x, max_y))
objects = [
    (0, (0, 0, 2, 2)),
    (1, (1, 1, 3, 3)),
    (2, (4, 4, 6, 6))
]

for obj_id, bbox in objects:
    idx.insert(obj_id, bbox)
    print(f"✓ Inserted {obj_id}: {bbox}")

# Query intersection
query_bbox = (1.5, 1.5, 3.5, 3.5)
results = list(idx.intersection(query_bbox))
print(f"✓ Intersection query {query_bbox}: {results}")
# Boxes 0 and 1 overlap the query box; box 2 lies outside it. intersection() has no fixed order.
assert sorted(results) == [0, 1], results
assert sorted(idx.intersection((0, 0, 1, 1))) == [0, 1]
assert list(idx.intersection((10, 10, 11, 11))) == []
assert idx.count((0, 0, 10, 10)) == 3

## Example 2: R-tree properties

In [ ]:
# Example 2: R-tree properties
from rtree import index

print("Testing rtree properties:")

# Create index with custom properties
props = index.Property()
props.dimension = 2
props.variant = index.RT_Star
print(f"✓ Properties created: dimension={props.dimension}, variant={props.variant}")

# Create index with properties
idx = index.Index(properties=props)
print(f"✓ Index with custom properties: {type(idx).__name__}")

# Insert test data
for i in range(5):
    bbox = (i, i, i+1, i+1)
    idx.insert(i, bbox)

print(f"✓ Inserted 5 objects")
print(f"✓ Index bounds: {idx.get_bounds()}")
assert idx.properties.dimension == 2 and idx.properties.variant == index.RT_Star
assert len(idx) == 5, len(idx)
# The bounds cover every inserted box: (min_x, min_y, max_x, max_y)
assert idx.get_bounds() == [0.0, 0.0, 5.0, 5.0], idx.get_bounds()

## Example 3: Nearest neighbor queries

In [ ]:
# Example 3: Nearest neighbor queries
from rtree import index

print("Testing rtree nearest neighbor:")

# Create index and insert objects
idx = index.Index()
objects = [(i, (i*10, i*10, i*10+5, i*10+5)) for i in range(5)]

for obj_id, bbox in objects:
    idx.insert(obj_id, bbox)

print(f"✓ Inserted {len(objects)} objects")

# Nearest neighbor query
query_point = (22, 22, 22, 22)
nearest = list(idx.nearest(query_point, 3))

print(f"✓ Query point: (22, 22)")
print(f"✓ 3 nearest objects: {nearest}")
# Box 2 (20..25) contains the point; box 1 (10..15) is about 9.9 away and box 3 (30..35) about 11.3.
assert nearest == [2, 1, 3], nearest
assert list(idx.nearest(query_point, 1)) == [2]

## Example 4: 3D spatial indexing

In [ ]:
# Example 4: 3D spatial indexing
from rtree import index

print("Testing rtree 3D indexing:")

# Create 3D index
props = index.Property()
props.dimension = 3
idx_3d = index.Index(properties=props)

print(f"✓ 3D index created: dimension={props.dimension}")

# Insert 3D objects (min_x, min_y, min_z, max_x, max_y, max_z)
objects_3d = [
    (0, (0, 0, 0, 5, 5, 5)),
    (1, (3, 3, 3, 8, 8, 8)),
    (2, (10, 10, 10, 15, 15, 15))
]

for obj_id, bbox in objects_3d:
    idx_3d.insert(obj_id, bbox)
    print(f"✓ Inserted 3D object {obj_id}: {bbox}")

# 3D query
query_3d = (4, 4, 4, 12, 12, 12)
results = list(idx_3d.intersection(query_3d))
print(f"✓ 3D intersection {query_3d}: {results}")
assert idx_3d.properties.dimension == 3
assert sorted(results) == [0, 1, 2], results
assert sorted(idx_3d.intersection((9, 9, 9, 9.5, 9.5, 9.5))) == []

## Example 5: Delete and update operations

In [ ]:
# Example 5: Delete and update operations
from rtree import index

print("Testing rtree delete and update:")

# Create index with data
idx = index.Index()
objects = [(i, (i, i, i+2, i+2)) for i in range(5)]

for obj_id, bbox in objects:
    idx.insert(obj_id, bbox)

print(f"✓ Initial size: {len(list(idx.intersection((-10, -10, 20, 20))))}")

# Delete an object
idx.delete(2, (2, 2, 4, 4))
print(f"✓ Deleted object 2")

# Update (delete + insert)
idx.delete(3, (3, 3, 5, 5))
idx.insert(3, (10, 10, 12, 12))
print(f"✓ Updated object 3 to new location")

# Final size
final_count = len(list(idx.intersection((-10, -10, 20, 20))))
print(f"✓ Final size: {final_count}")
assert final_count == 4, final_count
assert sorted(idx.intersection((-10, -10, 20, 20))) == [0, 1, 3, 4]
# Object 3 is found at its new location and no longer at the old one
assert list(idx.intersection((9, 9, 13, 13))) == [3]
assert 3 not in idx.intersection((3, 3, 3.5, 3.5))